# HA 3

### Problem 1 (1 point)

Diffie–Hellman key exchange protocol is one of the earliest practical examples of public key exchange implemented within the field of cryptography. Published in 1976 by Diffie and Hellman, this is the earliest publicly known work that proposed the idea of a private key and a corresponding public key. 

1) Implement function to generate common secret key within multiplicative group of given Finite field with known generator. 

*Note. You can assume that all the numbers are small, for example, less than $2^{16}$.*

In [ ]:
def diffie_hellman(p, g, alice_private_key, bob_public_key):
    alice_public_key = pow(g, alice_private_key, p) # g ^ alice_private_key % p
    shared_secret = pow(bob_public_key, alice_private_key, p) # bob_public_key ^ alice_private_key % p

    return alice_public_key, shared_secret

2) Test your solution in GF(17) with generator g=11. Bobs' open key B=11, Alice private key is a=7

In [24]:
p = 17
g = 11
B = 11
a = 7

A, shared_secret = diffie_hellman(p, g, a, B)

print("Alice public key A =", A)
print("Shared secret =", shared_secret)

Alice public key A = 3
Shared secret = 3


### Problem 2 (2 points)

El Gamal protocol is widely used in cryptography. In this task we will ask you to implement your own El-Gamal encryption scheme on Python.

1) Implement function for generating keys. The function must generate big random prime number (problem of generating big prime numbers was discussed within the lectures). (0.5 point)

*Note. You can assume that all the numbers are small, for example, less than $2^{32}$. But you **must** use a primality test as a part of the function to get a full score.*

In [25]:
import secrets

def is_prime(n):
    if n < 2:
        return False
    for prime in (2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37):
        if n == prime:
            return True
        if n % prime == 0:
            return False

    d, s = n - 1, 0
    while d % 2 == 0:
        d //= 2
        s += 1

    for a in (2, 3, 5, 7, 11):
        x = pow(a, d, n)
        if x == 1 or x == n - 1:
            continue
        for _ in range(s - 1):
            x = pow(x, 2, n)
            if x == n - 1:
                break
        else:
            return False
    return True

def random_prime(bits=17):
    while True:
        candidate = secrets.randbits(bits) | (1 << (bits - 1)) | 1
        if is_prime(candidate):
            return candidate

def prime_factors(n):
    factors = set()
    divisor = 2
    while divisor * divisor <= n:
        while n % divisor == 0:
            factors.add(divisor)
            n //= divisor
        divisor += 1
    if n > 1:
        factors.add(n)
    return factors

def primitive_root(p):
    # Find a generator g of the multiplicative group GF(p)*. 
    factors = prime_factors(p - 1)
    for g in range(2, p):
        if all(pow(g, (p - 1) // factor, p) != 1 for factor in factors):
            return g

def generate_elgamal_keys(bits=17):
    p = random_prime(bits)
    g = primitive_root(p)
    private_key = secrets.randbelow(p - 2) + 1
    public_key = pow(g, private_key, p)
    return (p, g, public_key), private_key

private key = random integer $x$.   
The public key is $(p, g, y)$, where $p$ is prime, $g$ is a generator of $GF(p)^*$, and $y=g^x \bmod p$.

2) Implement functions that realize the encryption and decryption in El Gamal protocol. (0.5 points)

In [26]:
def elgamal_encrypt(message, public_key):
    p, g, y = public_key

    k = secrets.randbelow(p - 2) + 1
    c1 = pow(g, k, p)
    c2 = (message * pow(y, k, p)) % p
    return c1, c2

def elgamal_decrypt(ciphertext, private_key, p):
    c1, c2 = ciphertext
    shared_secret = pow(c1, private_key, p)
    shared_secret_inverse = pow(shared_secret, -1, p)
    return (c2 * shared_secret_inverse) % p

To encrypt $m$: choose a fresh random $k$ and calculate $c_1=g^k \bmod p$ and $c_2=m y^k \bmod p$.   
To decrypt $m$: calculate $s=c_1^x=y^k \bmod p$ and multiply $c_2$ by $s^{-1}$ modulo $p$.



3) Calculate Hash of your name by SHA-1 and test El Gamal encryption/decryption functions on its 16-bit prefix. (1 points)

In [28]:
from hashlib import sha1

name = "My name"
digest = sha1(name.encode()).hexdigest()
message = int(digest[:4], 16)  # first 16 bits = first 4 hexadecimal characters. This is an integer representation

public_key, private_key = generate_elgamal_keys()
ciphertext = elgamal_encrypt(message, public_key)
decrypted_message = elgamal_decrypt(ciphertext, private_key, public_key[0])

print("SHA-1:", digest)
print("16-bit prefix:", message)
print("Ciphertext:", ciphertext)
print("Decrypted message:", decrypted_message)
print("Correct:", message == decrypted_message)

SHA-1: a8f40df7087b9a42dd15516402cd76003c7d33b5
16-bit prefix: 43252
Ciphertext: (49949, 47635)
Decrypted message: 43252
Correct: True


### Problem 3 (3 points)

Elliptic curves due to their efficient hardware realization widely used in modern secure communication channels. The main thing that lies inside their cryptographic hardness is that we can break them only by greed search over all group points. In this task, we will ask you to write Python function that returns all group elements of a certain elliptic curve over a finite field 

1) Write a python function that list all points of elliptic curve $y^2=x^3+7$ over $F_{127}$ (0.5 point)

*Note. $127 = 2^7-1$ is the fourth Mersenne prime.*

In [29]:
p = 127

def curve_points(p): 
    # brute force 
    points = []
    for x in range(p):
        for y in range(p):
            if (y ** 2 - x ** 3 - 7) % p == 0:
                points.append((x, y))
    return points

points = curve_points(p)
print(points)
print("Number of affine points:", len(points))

[(1, 32), (1, 95), (2, 53), (2, 74), (3, 62), (3, 65), (4, 43), (4, 84), (8, 30), (8, 97), (11, 24), (11, 103), (12, 46), (12, 81), (14, 46), (14, 81), (17, 27), (17, 100), (18, 39), (18, 88), (19, 32), (19, 95), (21, 39), (21, 88), (24, 49), (24, 78), (25, 30), (25, 97), (28, 49), (28, 78), (32, 3), (32, 124), (34, 24), (34, 103), (38, 53), (38, 74), (39, 12), (39, 115), (41, 27), (41, 100), (45, 33), (45, 94), (46, 51), (46, 76), (47, 43), (47, 84), (51, 18), (51, 109), (52, 36), (52, 91), (57, 62), (57, 65), (58, 38), (58, 89), (60, 19), (60, 108), (67, 62), (67, 65), (69, 27), (69, 100), (70, 19), (70, 108), (71, 63), (71, 64), (72, 16), (72, 111), (75, 49), (75, 78), (76, 43), (76, 84), (78, 50), (78, 77), (79, 63), (79, 64), (80, 18), (80, 109), (82, 24), (82, 103), (84, 16), (84, 111), (85, 50), (85, 77), (86, 38), (86, 89), (87, 53), (87, 74), (88, 39), (88, 88), (91, 50), (91, 77), (93, 33), (93, 94), (94, 30), (94, 97), (96, 51), (96, 76), (98, 16), (98, 111), (99, 36), (99, 

2) Compare the number of points with Hasse’s estimate $|N-(q+1)|\leq 2{\sqrt  {q}}$. (0.5 point)

In [30]:
N = len(points) + 1 # Adding the point at infinity to count all group points
q = 127

left = abs(N - (q + 1))
right = 2 * (q ** 0.5)

print("N =", N)
print("|N - (q + 1)| =", left)
print("2 * sqrt(q) =", right)
print("Hasse estimate is true:", left <= right)

N = 127
|N - (q + 1)| = 1
2 * sqrt(q) = 22.538855339169288
Hasse estimate is true: True


3) Prove that the point
$A = (19, 32)$ belongs to the elliptic curve and construct a sequence of $B_n = nA, n = 1, ..., 100$. (2 points)

In [31]:
A = (19, 32)

# Check that A belongs to the curve
print("y^2 mod 127 =", A[1] ** 2 % p)
print("x^3 + 7 mod 127 =", (A[0] ** 3 + 7) % p) 
print("Does A belong to the elliptic curve? Answer:", A[1] ** 2 % p == (A[0] ** 3 + 7) % p)

def add_points(P, Q, p):
    if P is None:
        return Q
    if Q is None:
        return P

    x1, y1 = P
    x2, y2 = Q

    # P + (-P) is the same point at infinity
    if x1 == x2 and (y1 + y2) % p == 0:
        return None

    if P == Q:
        slope = (3 * x1 ** 2 * pow(2 * y1, -1, p)) % p
    else:
        slope = ((y2 - y1) * pow(x2 - x1, -1, p)) % p

    x3 = (slope ** 2 - x1 - x2) % p
    y3 = (slope * (x1 - x3) - y1) % p
    return x3, y3

B = None
for n in range(1, 101):
    B = add_points(B, A, p)
    print(f"B_{n} = {B}")

y^2 mod 127 = 8
x^3 + 7 mod 127 = 8
Does A belong to the elliptic curve? Answer: True
B_1 = (19, 32)
B_2 = (11, 24)
B_3 = (98, 16)
B_4 = (123, 18)
B_5 = (79, 64)
B_6 = (110, 38)
B_7 = (58, 38)
B_8 = (38, 53)
B_9 = (67, 62)
B_10 = (91, 50)
B_11 = (25, 30)
B_12 = (69, 27)
B_13 = (86, 89)
B_14 = (41, 27)
B_15 = (103, 91)
B_16 = (122, 3)
B_17 = (60, 19)
B_18 = (24, 78)
B_19 = (1, 32)
B_20 = (107, 95)
B_21 = (75, 78)
B_22 = (109, 12)
B_23 = (21, 39)
B_24 = (4, 84)
B_25 = (2, 53)
B_26 = (17, 100)
B_27 = (104, 64)
B_28 = (76, 84)
B_29 = (96, 76)
B_30 = (116, 94)
B_31 = (99, 91)
B_32 = (71, 63)
B_33 = (87, 74)
B_34 = (94, 30)
B_35 = (45, 94)
B_36 = (101, 46)
B_37 = (106, 12)
B_38 = (34, 24)
B_39 = (28, 49)
B_40 = (82, 103)
B_41 = (124, 19)
B_42 = (14, 46)
B_43 = (112, 76)
B_44 = (57, 62)
B_45 = (8, 97)
B_46 = (52, 36)
B_47 = (51, 18)
B_48 = (18, 39)
B_49 = (12, 46)
B_50 = (100, 3)
B_51 = (80, 109)
B_52 = (47, 43)
B_53 = (3, 65)
B_54 = (78, 50)
B_55 = (46, 76)
B_56 = (88, 39)
B_57 = (72, 16)
B_

### Problem 4 (1 point)

Let $p = 601$, $q = 7$, $e = 1463$ be the setup of the RSA algorithm. Compute $d$, sign the plane message $m = 58$ and check the signature.

In [32]:
p = 601
q = 7
e = 1463
m = 58

n = p * q
phi = (p - 1) * (q - 1)
d = pow(e, -1, phi)  # e * d = 1 mod phi

signature = pow(m, d, n)
checked_message = pow(signature, e, n)

print("n =", n)
print("phi(n) =", phi)
print("d =", d)
print("Signature =", signature)
print("Verified message =", checked_message)
print("Signature is valid:", checked_message == m)

n = 4207
phi(n) = 3600
d = 1127
Signature = 1348
Verified message = 58
Signature is valid: True


### Problem 5 (3 points)

The dealer shared the secret among four participants using a (n, 4)-threshold Shamir secret sharing scheme over ( F_{19} ) with the following shares: ( (3, 17), (17, 5), (5, 11), (11, 4) ).

1) (1 point) Find the secret (the value at ( x=0 )) and the initial polynomial. Is it consistent with the (4, 3) threshold?

***Hint.*** *Refer to whiteboard 07.3 for an example.*

In [34]:
p = 19
shares = [(3, 17), (17, 5), (5, 11), (11, 4)]

# Lagrange interpolation
def lagrange(x, shares, p):
    result = 0
    for xi, yi in shares:
        term = yi
        for xj, _ in shares:
            if xi != xj:
                term = term * (x - xj) * pow(xi - xj, -1, p) % p
        result = (result + term) % p
    return result

secret = lagrange(0, shares, p)
print("Secret f(0):", secret)
print("Polynomial: f(x) = x^3 + 3*x^2 + 1 mod 19")

for x, y in shares:
    print(f"f({x}) = {lagrange(x, shares, p)}")

Secret f(0): 1
Polynomial: f(x) = x^3 + 3*x^2 + 1 mod 19
f(3) = 17
f(17) = 5
f(5) = 11
f(11) = 4


It is not a $(4,3)$-threshold scheme, because a $(4,3)$ scheme uses a polynomial of degree at most 2.

2) (2 points) Provide a probabilistic secret generation procedure for the aforementioned example and theoretically prove that it ensures perfect secrecy.

In [35]:
import secrets

def create_shares(secret, x_values, p=19):
    # Random degree-3 polynomial with f(0) = secret
    a1 = secrets.randbelow(p)
    a2 = secrets.randbelow(p)
    a3 = secrets.randbelow(p)

    def f(x):
        return (secret + a1 * x + a2 * x ** 2 + a3 * x ** 3) % p

    return [(x, f(x)) for x in x_values]

new_shares = create_shares(1, [3, 17, 5, 11])
print(new_shares)

[(3, 9), (17, 14), (5, 10), (11, 2)]


The coefficients $a_1,a_2,a_3$ are chosen independently and uniformly from $F_{19}$.   

With fewer than four shares, any candidate secret has exactly one choice of coefficients that produces the given shares. 

That's why every secret is equally likely after seeing fewer than four shares, so the scheme has perfect secrecy.